# Homework: Numerical Integration

## Problem 1: Implementing and Comparing Quadrature Rules

**(a)** Implement the composite trapezoidal rule and Simpson's rule with the following signatures:

In [1]:
import numpy as np
from scipy import stats
def trapezoidal(f, a, b, n):
    """Composite trapezoidal rule with n subintervals.

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of subintervals.

    Returns
    -------
    float
        Approximate integral value.
    """
    if n <= 0:
        raise ValueError("n must be a positive integer.")

    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)

    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def simpsons(f, a, b, n):
    """Composite Simpson's rule with n subintervals (n must be even).

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of subintervals (must be even).

    Returns
    -------
    float
        Approximate integral value.
    """
    if n <= 0:
        raise ValueError("n must be a positive integer.")
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's rule.")

    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)

    return (h / 3) * (
        y[0]
        + 4 * np.sum(y[1:-1:2])
        + 2 * np.sum(y[2:-1:2])
        + y[-1]
    )

In [2]:

# Define function
f = lambda x: np.exp(-x**2)

# "True" value (high precision)
I_true = 0.7468241328124271

# n values
n_values = [4, 16, 64, 256]

print("n | Trap Approx | Trap Error | Simpson Approx | Simpson Error")
print("-" * 70)

for n in n_values:
    trap_val = trapezoidal(f, 0, 1, n)
    simp_val = simpsons(f, 0, 1, n)
    trap_err = abs(trap_val - I_true)
    simp_err = abs(simp_val - I_true)

    print(f"{n:<3} | {trap_val:.10f} | {trap_err:.2e} | {simp_val:.10f} | {simp_err:.2e}")


n | Trap Approx | Trap Error | Simpson Approx | Simpson Error
----------------------------------------------------------------------
4   | 0.7429840978 | 3.84e-03 | 0.7468553798 | 3.12e-05
16  | 0.7465845968 | 2.40e-04 | 0.7468242574 | 1.25e-07
64  | 0.7468091636 | 1.50e-05 | 0.7468241333 | 4.87e-10
256 | 0.7468231972 | 9.36e-07 | 0.7468241328 | 1.90e-12


Test both on $I = \int_0^1 e^{-x^2} dx$ with $n = 4, 16, 64, 256$. For each $n$, report the absolute error.

**(b)** For each method, verify the theoretical convergence rate by computing the ratio of consecutive errors as $n$ doubles. The trapezoidal rule should show a ratio near 4 (since error is $O(n^{-2})$) and Simpson's rule should show a ratio near 16 (since error is $O(n^{-4})$).



In [3]:
trap_errors = []
simp_errors = []

for n in n_values:
    trap_val = trapezoidal(f, 0, 1, n)
    simp_val = simpsons(f, 0, 1, n)

    trap_errors.append(abs(trap_val - I_true))
    simp_errors.append(abs(simp_val - I_true))

print("n values:", n_values)
print("Trapezoidal errors:", trap_errors)
print("Simpson errors:", simp_errors)

print("\nConsecutive error ratios:")
for i in range(len(n_values) - 1):
    n1 = n_values[i]
    n2 = n_values[i + 1]

    trap_ratio = trap_errors[i] / trap_errors[i + 1]
    simp_ratio = simp_errors[i] / simp_errors[i + 1]

    print(f"{n1} -> {n2}:")
    print(f"  Trapezoidal ratio = {trap_ratio:.4f}")
    print(f"  Simpson ratio     = {simp_ratio:.4f}")

n values: [4, 16, 64, 256]
Trapezoidal errors: [np.float64(0.0038400350120458837), np.float64(0.0002395360242054556), np.float64(1.496917459897773e-05), np.float64(9.355662747845273e-07)]
Simpson errors: [np.float64(3.124697856016212e-05), np.float64(1.2462330323259607e-07), np.float64(4.872454661963843e-10), np.float64(1.9033663534173684e-12)]

Consecutive error ratios:
4 -> 16:
  Trapezoidal ratio = 16.0311
  Simpson ratio     = 250.7314
16 -> 64:
  Trapezoidal ratio = 16.0020
  Simpson ratio     = 255.7711
64 -> 256:
  Trapezoidal ratio = 16.0001
  Simpson ratio     = 255.9914


## Problem 2: Variance Reduction with Antithetic Variables

Standard Monte Carlo integration draws independent uniform samples to estimate an integral. Antithetic variables is a variance reduction technique that pairs each sample $U_i$ with its "mirror" $a + b - U_i$ on the interval $[a, b]$. When the integrand is monotone, these pairs are negatively correlated, which reduces the variance of the estimator.

**(a)** Implement a Monte Carlo integrator that uses antithetic variables. For each of $n$ uniform draws $U_i$ on $[a, b]$, compute both $f(U_i)$ and $f(a + b - U_i)$, then average the pair. The estimate is the mean of these $n$ pairwise averages (using $2n$ total function evaluations).

In [4]:
import numpy as np

def mc_antithetic(f, a, b, n, seed=None):
    """Monte Carlo integration with antithetic variables on [a, b].

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of antithetic pairs (total function evaluations = 2n).
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    estimate : float
        Antithetic MC estimate of the integral.
    se : float
        Standard error of the estimate.
    """
    rng = np.random.default_rng(seed)

    u = rng.uniform(a, b, size=n)
    u_mirror = a + b - u

    # Pairwise avg
    pair_avg = 0.5 * (f(u) + f(u_mirror))

    estimate = (b - a) * np.mean(pair_avg)
    se = (b - a) * np.std(pair_avg, ddof=1) / np.sqrt(n)

    return estimate, se

In [5]:

I_true = 0.7468241328124271

for n in [500, 5000, 50000]:
    est, se = mc_antithetic(f, 0, 1, n, seed=42)
    err = abs(est - I_true)
    print(f"n = {n:6d} | estimate = {est:.10f} | SE = {se:.10e} | abs error = {err:.10e}")

n =    500 | estimate = 0.7472814759 | SE = 1.2244694804e-03 | abs error = 4.5734303999e-04
n =   5000 | estimate = 0.7468536627 | SE = 3.9994889091e-04 | abs error = 2.9529884203e-05
n =  50000 | estimate = 0.7468361591 | SE = 1.2689278766e-04 | abs error = 1.2026246631e-05


Test on $I = \int_0^1 e^{-x^2} dx$ with $n = 500, 5000, 50000$ pairs (use `seed=42` with `numpy.random.default_rng`). For each $n$, report the estimate, standard error, and absolute error.

**(b)** For a fair comparison, standard MC with the same computational budget uses $2n$ independent samples. For each value of $n$ above, also compute the standard MC estimate and SE using $2n$ samples (same seed). Report the variance reduction factor $\text{SE}_{\text{standard}}^2 / \text{SE}_{\text{antithetic}}^2$ for each $n$.



In [6]:
def mc_standard(f, a, b, m, seed=None):
    """Standard Monte Carlo integration on [a, b] with m samples."""
    rng = np.random.default_rng(seed)
    u = rng.uniform(a, b, size=m)
    vals = f(u)

    estimate = (b - a) * np.mean(vals)
    se = (b - a) * np.std(vals, ddof=1) / np.sqrt(m)

    return estimate, se


for n in [500, 5000, 50000]:
    est_a, se_a = mc_antithetic(f, 0, 1, n, seed=42)
    est_s, se_s = mc_standard(f, 0, 1, 2*n, seed=42)

    err_a = abs(est_a - I_true)
    err_s = abs(est_s - I_true)
    vrf = (se_s**2) / (se_a**2)

    print(f"\nn = {n}")
    print(f"Antithetic: estimate = {est_a:.10f}, SE = {se_a:.10e}, abs error = {err_a:.10e}")
    print(f"Standard  : estimate = {est_s:.10f}, SE = {se_s:.10e}, abs error = {err_s:.10e}")
    print(f"Variance reduction factor = {vrf:.4f}")


n = 500
Antithetic: estimate = 0.7472814759, SE = 1.2244694804e-03, abs error = 4.5734303999e-04
Standard  : estimate = 0.7479987407, SE = 6.4015695499e-03, abs error = 1.1746078836e-03
Variance reduction factor = 27.3324

n = 5000
Antithetic: estimate = 0.7468536627, SE = 3.9994889091e-04, abs error = 2.9529884203e-05
Standard  : estimate = 0.7489294663, SE = 2.0050417183e-03, abs error = 2.1053334856e-03
Variance reduction factor = 25.1326

n = 50000
Antithetic: estimate = 0.7468361591, SE = 1.2689278766e-04, abs error = 1.2026246631e-05
Standard  : estimate = 0.7463959333, SE = 6.3516179226e-04, abs error = 4.2819950118e-04
Variance reduction factor = 25.0550


**(c)** Explain why antithetic variables reduce variance for the integrand $e^{-x^2}$ on $[0, 1]$. Describe a type of function for which antithetic variables would provide little or no variance reduction.

The function is monotone decreasing. So if U is randomly drawn, then
f(U) and f(1−U) tend to move in opposite directions: when one is large, the other is small. This creates negative correlation, which reduces the variance of the averaged pair and therefore lowers the variance of the estimator.

## Problem 3: Monte Carlo Integration

**(a)** Write a function that estimates $\int_a^b f(x) dx$ by Monte Carlo integration and returns both the estimate and its standard error:

In [7]:
def mc_integrate(f, a, b, n, seed=None):
    """Monte Carlo integration on [a, b].

    Parameters
    ----------
    f : callable
        Function to integrate.
    a, b : float
        Integration limits.
    n : int
        Number of random samples.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    estimate : float
        MC estimate of the integral.
    se : float
        Standard error of the estimate.
    """
    rng = np.random.default_rng(seed)
    u = rng.uniform(a, b, size=n)
    vals = f(u)

    estimate = (b - a) * np.mean(vals)
    se = (b - a) * np.std(vals, ddof=1) / np.sqrt(n)

    return estimate, se

In [8]:
n_values = [100, 1000, 10000, 100000]

results = []

for n in n_values:
    est, se = mc_integrate(f, 0, 1, n, seed=42)
    err = abs(est - I_true)
    results.append((n, est, se, err))
    print(f"n = {n:6d} | estimate = {est:.10f} | SE = {se:.10e} | abs error = {err:.10e}")

n =    100 | estimate = 0.7583531360 | SE = 1.8883290175e-02 | abs error = 1.1529003159e-02
n =   1000 | estimate = 0.7479987407 | SE = 6.4015695499e-03 | abs error = 1.1746078836e-03
n =  10000 | estimate = 0.7489294663 | SE = 2.0050417183e-03 | abs error = 2.1053334856e-03
n = 100000 | estimate = 0.7463959333 | SE = 6.3516179226e-04 | abs error = 4.2819950118e-04


Test on $\int_0^1 e^{-x^2} dx$ with $n = 100, 1000, 10000, 100000$ (use `seed=42`). For each $n$, report the estimate, standard error, and absolute error.

**(b)** Verify that the MC error rate is $O(n^{-1/2})$: compute the ratio of standard errors as $n$ increases by a factor of 100 (from 100 to 10000). The SE should decrease by a factor of approximately 10.


In [9]:
for i in range(len(results) - 1):
    n1, _, se1, _ = results[i]
    n2, _, se2, _ = results[i + 1]
    print(f"{n1} -> {n2}: SE ratio = {se1 / se2:.4f}")

100 -> 1000: SE ratio = 2.9498
1000 -> 10000: SE ratio = 3.1927
10000 -> 100000: SE ratio = 3.1567



**(c)** The $O(n^{-1/2})$ rate appears slow compared to deterministic quadrature. With 256 subintervals, Simpson's rule achieves error around $10^{-12}$ for this 1D integral, while MC with 100,000 points still has error around $10^{-4}$. Why then is MC preferred for high-dimensional integrals?

Although Monte Carlo converges slowly, its rate is independent of dimension. In contrast, deterministic methods suffer from the curse of dimensionality, with cost growing exponentially as dimension increases. Thus, while deterministic methods are more accurate in low dimensions, Monte Carlo is preferred in high-dimensional settings.

## Problem 4: Importance Sampling

The following code estimates $P(Z > 4)$ where $Z \sim N(0,1)$ using naive Monte Carlo and importance sampling with a shifted exponential proposal. Read the code and answer the questions.

In [10]:
import numpy as np
from scipy import stats

true_prob = 1 - stats.norm.cdf(4)  # ≈ 3.17e-5

# Naive MC
np.random.seed(42)
z_samples = np.random.normal(0, 1, 100000)
naive_est = np.mean(z_samples > 4)

# Importance sampling with Exp(1) shifted to start at 4
np.random.seed(42)
x_is = np.random.exponential(1.0, 100000) + 4.0
log_weights = stats.norm.logpdf(x_is) - (-1.0 * (x_is - 4))
weights = np.exp(log_weights)
is_est = np.mean(weights)

**(a)** Explain why naive MC performs poorly for this problem. How many of the 100,000 standard normal samples are expected to exceed 4?

This quite rare-event probability. The expected count is 3.17

**(b)** The importance sampling proposal is $q(x) = e^{-(x-4)}$ for $x \geq 4$.  
Verify that `log_weights` correctly computes $\log[\phi(x)/q(x)]$ where $\phi$ is the standard normal density.

$
\log \frac{\phi(x)}{q(x)}
= \log \phi(x) - \log q(x)
= \log \phi(x) - \bigl( -(x - 4) \bigr)
= \log \phi(x) + (x - 4)
$

**(c)** A classmate suggests using a $t$-distribution with 3 degrees of freedom as the proposal instead of the shifted exponential. Would this be a good choice for estimating $P(Z > 4)$? Why or why not?

$t_3$ would usually not be a good choice here. Its tails are much heavier than the normal tail, so it would generate many values far above 4 where the normal density is extremely small. That makes the importance weights more variable and reduces efficiency.

The shifted exponential is better because it places samples directly in the target tail region x≥4 without being unnecessarily heavy-tailed.

**(d)** Another classmate proposes using $N(5, 0.5^2)$ as the proposal. Compute the effective sample size (ESS) and compare it to using the shifted exponential. Which proposal is better and why?


The shifted exponential proposal is better because it concentrates all samples in the relevant tail region x≥4 and produces less variable importance weights, leading to a much larger ESS and a more efficient estimator.

In [11]:
# Template for part (d)
np.random.seed(42)
n_is = 10000

# Shifted exponential proposal (all samples are >= 4 by construction)
x_exp = np.random.exponential(1.0, n_is) + 4.0
log_w_exp = stats.norm.logpdf(x_exp) + (x_exp - 4)
log_w_exp -= np.max(log_w_exp)
w_exp = np.exp(log_w_exp)
w_exp_norm = w_exp / np.sum(w_exp)
ess_exp = 1.0 / np.sum(w_exp_norm ** 2)

# N(5, 0.5^2) proposal: multiply weights by indicator I(x > 4)
x_norm = np.random.normal(5, 0.5, n_is)
log_w_norm = stats.norm.logpdf(x_norm) - stats.norm.logpdf(x_norm, 5, 0.5)
w_norm_raw = np.exp(log_w_norm) * (x_norm > 4)
w_norm_raw[w_norm_raw > 0] /= np.max(w_norm_raw[w_norm_raw > 0])
w_norm_norm = w_norm_raw / np.sum(w_norm_raw)
ess_norm = 1.0 / np.sum(w_norm_norm ** 2)
print(f"ess_norm {ess_norm}")
print(f"ess_exp {ess_exp}")

ess_norm 677.4226510447006
ess_exp 4137.000460254567


## Problem 5: The Curse of Dimensionality

Deterministic quadrature rules extend to multiple dimensions via product grids. For a $d$-dimensional integral with $n$ nodes per dimension, the total number of function evaluations is $n^d$. Monte Carlo, by contrast, uses a fixed sample size regardless of dimension. In this problem, you will observe the crossover point where MC becomes more efficient than product-rule quadrature.

Consider the integral $I_d = \int_{[0,1]^d} e^{-\|\mathbf{x}\|^2} \, d\mathbf{x}$ where $\|\mathbf{x}\|^2 = x_1^2 + \cdots + x_d^2$.

**(a)** Write a function that computes $I_d$ using a product trapezoidal rule with $n$ nodes per dimension.

In [12]:
import numpy as np

def trap_nd(f, d, n):
    """Product trapezoidal rule on [0,1]^d with n nodes per dimension.

    Parameters
    ----------
    f : callable
        Function from R^d to R. Takes an array of shape (N, d)
        where N is the number of evaluation points.
    d : int
        Number of dimensions.
    n : int
        Number of nodes per dimension.

    Returns
    -------
    float
        Approximate integral value.
    """
    x = np.linspace(0.0, 1.0, n)
    h = 1.0 / (n - 1)

    # full tensor-product grid, shape (n^d, d)
    meshes = np.meshgrid(*([x] * d), indexing="ij")
    pts = np.stack([m.ravel() for m in meshes], axis=1)

    # trapezoidal weights in 1D: 1/2 at endpoints, 1 otherwise
    w1 = np.ones(n)
    w1[0] = 0.5
    w1[-1] = 0.5

    # product weights for d dimensions
    w_meshes = np.meshgrid(*([w1] * d), indexing="ij")
    w = np.prod(np.stack(w_meshes, axis=0), axis=0).ravel()

    return (h ** d) * np.sum(w * f(pts))
def f(x):
    return np.exp(-np.sum(x**2, axis=1))


import numpy as np
from scipy.integrate import quad

def trap_nd(f, d, n):
    x = np.linspace(0.0, 1.0, n)
    h = 1.0 / (n - 1)

    meshes = np.meshgrid(*([x] * d), indexing="ij")
    pts = np.stack([m.ravel() for m in meshes], axis=1)

    w1 = np.ones(n)
    w1[0] = 0.5
    w1[-1] = 0.5

    w_meshes = np.meshgrid(*([w1] * d), indexing="ij")
    w = np.prod(np.stack(w_meshes, axis=0), axis=0).ravel()

    return (h ** d) * np.sum(w * f(pts))

def f(x):
    return np.exp(-np.sum(x**2, axis=1))

I1, _ = quad(lambda x: np.exp(-x**2), 0, 1)

n = 10
rng = np.random.default_rng(42)

print(" d | evals | trap_est | exact | trap_err | mc_est | mc_err")
print("-" * 65)

for d in range(1, 7):
    exact = I1 ** d
    evals = n ** d

    trap_est = trap_nd(f, d, n)
    trap_err = abs(trap_est - exact)

    # MC
    x_mc = rng.uniform(0.0, 1.0, size=(evals, d))
    mc_est = np.mean(f(x_mc))
    mc_err = abs(mc_est - exact)

    print(f"{d:2d} | {evals:5d} | {trap_est:.8f} | {exact:.8f} | "
          f"{trap_err:.3e} | {mc_est:.8f} | {mc_err:.3e}")

 d | evals | trap_est | exact | trap_err | mc_est | mc_err
-----------------------------------------------------------------
 1 |    10 | 0.74606687 | 0.74682413 | 7.573e-04 | 0.67441688 | 7.241e-02
 2 |   100 | 0.55661577 | 0.55774629 | 1.131e-03 | 0.57852653 | 2.078e-02
 3 |  1000 | 0.41527259 | 0.41653839 | 1.266e-03 | 0.41696269 | 4.243e-04
 4 | 10000 | 0.30982112 | 0.31108092 | 1.260e-03 | 0.30869172 | 2.389e-03
 5 | 100000 | 0.23114727 | 0.23232274 | 1.175e-03 | 0.23302967 | 7.069e-04
 6 | 1000000 | 0.17245132 | 0.17350423 | 1.053e-03 | 0.17349607 | 8.155e-06


The integrand separates as $e^{-\|\mathbf{x}\|^2} = \prod_{j=1}^d e^{-x_j^2}$, so the exact value is $I_d = \left(\int_0^1 e^{-x^2} dx\right)^d$. Use `scipy.integrate.quad` to compute the 1D reference value, then raise it to the $d$-th power. Compute the product trapezoidal estimate for $d = 1, 2, 3, 4, 5, 6$ with $n = 10$ nodes per dimension. For each $d$, report the number of function evaluations and the absolute error.

**(b)** For each dimension $d$ above, estimate $I_d$ using Monte Carlo with the same number of function evaluations as the product trapezoidal rule (i.e., $n^d$ MC samples). Use `seed=42` with `numpy.random.default_rng`. Compare the MC error to the trapezoidal error for each $d$. At what dimension does MC become more accurate?

Monte Carlo becomes more accurate starting at d = 3 (first crossover point), although there is a small fluctuation at d=4. For higher dimensions (especially d≥5), MC clearly outperforms the trapezoidal rule.

**(c)** The separability $e^{-\|\mathbf{x}\|^2} = \prod_j e^{-x_j^2}$ means one could compute the $d$-dimensional integral as a product of $d$ one-dimensional integrals, each requiring only $n$ nodes, for a total of $n \cdot d$ evaluations instead of $n^d$. Explain why this trick does not generalize: give an example of a non-separable integrand on $[0,1]^d$ that cannot be decomposed this way, and explain why such integrands are common in statistical applications involving correlated random effects.

Example could be: $f(x_1, x_2) = e^{-(x_1 + x_2)^2}$.

This function cannot be written as a product of one-dimensional functions $g_1(x_1)g_2(x_2)$, because expanding the exponent gives $(x_1 + x_2)^2 = x_1^2 + x_2^2 + 2x_1 x_2$
which contains a cross-term $2x_1 x_2$ that couples the variables.

Such non-separable integrands commonly arise in statistical applications involving correlated random effects, where joint densities contain quadratic forms such as $x^\top \Sigma^{-1} x$, introducing dependence between coordinates.